# Chapter 14: Time Series Classification — Companion Notebook

Runs **all** code blocks end-to-end (including those marked `eval: false` in the .qmd) to capture predictions, compute summary tables with timing, and save figures.

**Run from the `das_buch/` directory.**

## Imports and Setup

In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import PercentFormatter
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from scipy.spatial.distance import euclidean
import seaborn as sns
import json
import os

from datetime import datetime
from pathlib import Path
from IPython.display import display, HTML, Markdown

from aeon.distances import dtw_distance, msm_distance, euclidean_distance, wdtw_distance

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)

custom_palette = ["#000000", "#0072B2", "#D55E00", "#009E73", "#CC79A7", "#56B4E9", "#E69F00"]
line_styles = ['-', '--', '-.', ':']

class CFG:
    data_folder = Path.cwd().parent / "das_buch" / "data"
    img_dim1 = 12
    img_dim2 = 6
    fontsize = 18

plt.rcParams["figure.autolayout"] = True
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Source Sans Pro', 'Arial']
plt.rcParams['font.size'] = 14
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.titlesize'] = 18
plt.rcParams.update({'figure.figsize': (CFG.img_dim1, CFG.img_dim2)})

os.makedirs('images/chap14_images', exist_ok=True)

results = {}
timings = {}

# Auto-save figures: set _fig_save_path[0] before a function that calls plt.show() internally
_fig_save_path = [None]
_orig_plt_show = plt.show

def _patched_show(*args, **kwargs):
    if _fig_save_path[0]:
        plt.savefig(_fig_save_path[0], dpi=150, bbox_inches='tight')
        _fig_save_path[0] = None
    _orig_plt_show(*args, **kwargs)

plt.show = _patched_show


## Helper Functions

In [ ]:
def load_m5_subset(data_folder: Path) -> pd.DataFrame:
    filepath = data_folder / 'M5_t20_ABC.csv'
    if not filepath.exists():
        raise FileNotFoundError(f"Data file not found at {filepath}")
    df = pd.read_csv(filepath, index_col=0)
    df['date'] = pd.to_datetime(df['date'])
    df_sorted = df.sort_values(['item_id', 'date'])
    def remove_tail_values(group):
        return group.iloc[:-28]
    df = df_sorted.groupby('item_id').apply(remove_tail_values).reset_index(drop=True)
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['dayofweek'] = df['date'].dt.dayofweek
    df['quarter'] = df['date'].dt.quarter
    df['week_of_year'] = df['date'].dt.isocalendar().week
    return df


def analyze_dataset(df: pd.DataFrame) -> dict:
    abc_dist = df.groupby('item_id')['ABC_class'].first().value_counts().to_dict()
    analysis = {
        'n_series': df['item_id'].nunique(),
        'departments': df['dept_id'].nunique(),
        ''
        'date_range': (df['date'].min(), df['date'].max()),
        'total_sales': df['sold'].sum(),
        'mean_price': df['sell_price'].mean(),
        'abc_distribution': abc_dist,
        'sales_by_class': df.groupby('ABC_class')['sold'].sum().to_dict(),
        'avg_price_by_class': df.groupby('ABC_class')['sell_price'].mean().to_dict()
    }
    return analysis


def plot_sales_analysis(series: pd.DataFrame, figsize=(15, 10)):
    fig, axes = plt.subplots(2, 1, figsize=figsize)
    axes[0].plot(series['date'], series['sold'], alpha=0.7)
    axes[0].set_title(f"Sales Over Time for {series['item_id'].iloc[0]}")
    axes[0].set_xlabel('Date')
    axes[0].set_ylabel('Units Sold')
    sns.boxplot(data=series, x='dayofweek', y='sold', ax=axes[1])
    axes[1].set_title('Sales Distribution by Day of Week')
    axes[1].set_xlabel('Day of Week')
    axes[1].set_ylabel('Units Sold')
    plt.tight_layout()
    plt.show()
    print("\nSummary Statistics: " + str(series['sold'].describe()))


def plot_time_series(t, series_list, labels, title="Time Series Comparison"):
    plt.figure(figsize=(10, 6))
    for i, series in enumerate(series_list):
        plt.plot(t, series, label=labels[i], color=custom_palette[i % len(custom_palette)])
    plt.title(title, fontsize=14)
    plt.xlabel("Time", fontsize=12)
    plt.ylabel("Value", fontsize=12)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def calculate_distances(s1, s2):
    ed = euclidean_distance(s1, s2)
    dtw_dist = dtw_distance(s1, s2, window=None)
    dtw_window1 = dtw_distance(s1, s2, window=0.05)
    dtw_window2 = dtw_distance(s1, s2, window=0.10)
    wdtw_dist = wdtw_distance(s1, s2, g=0.1)
    return {
        "Euclidean": ed,
        "DTW": dtw_dist,
        "DTW (window=5%)": dtw_window1,
        "DTW (window=10%)": dtw_window2,
        "WDTW": wdtw_dist
    }


def visualize_euclidean_matching(t, s1, s2, title="Euclidean Distance: Point-to-Point Alignment"):
    plt.figure(figsize=(12, 6))
    plt.plot(t, s1, label="Series 1", color=custom_palette[0])
    plt.plot(t, s2, label="Series 2", color=custom_palette[1])
    for i in range(0, len(t), 10):
        plt.plot([t[i], t[i]], [s1[i], s2[i]], 'k--', alpha=0.5)
    indices = [10, 30, 50, 70, 90]
    for i in indices:
        plt.scatter(t[i], s1[i], color='red', s=50, zorder=5)
        plt.scatter(t[i], s2[i], color='red', s=50, zorder=5)
    ed = euclidean_distance(s1, s2)
    plt.text(0.02, 0.95, f"Euclidean Distance: {ed:.2f}", transform=plt.gca().transAxes,
             bbox=dict(facecolor='white', alpha=0.8), fontsize=12)
    plt.title(title, fontsize=14)
    plt.xlabel("Time", fontsize=12)
    plt.ylabel("Value", fontsize=12)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def visualize_dtw_matching(t, s1, s2, window=None, title="DTW: Optimal Path Alignment"):
    from scipy.spatial.distance import cdist
    s1_reshaped = s1.reshape(-1, 1)
    s2_reshaped = s2.reshape(-1, 1)
    cost_matrix = cdist(s1_reshaped, s2_reshaped, 'euclidean')
    acc_cost = np.zeros_like(cost_matrix)
    acc_cost[0, 0] = cost_matrix[0, 0]
    for i in range(1, len(s1)):
        acc_cost[i, 0] = acc_cost[i-1, 0] + cost_matrix[i, 0]
    for j in range(1, len(s2)):
        acc_cost[0, j] = acc_cost[0, j-1] + cost_matrix[0, j]
    if window is not None:
        window_size = int(window * len(s1)) if isinstance(window, float) and 0 <= window <= 1 else window
        for i in range(1, len(s1)):
            for j in range(1, len(s2)):
                if abs(i - j) <= window_size:
                    acc_cost[i, j] = cost_matrix[i, j] + min(acc_cost[i-1, j], acc_cost[i, j-1], acc_cost[i-1, j-1])
                else:
                    acc_cost[i, j] = np.inf
    else:
        for i in range(1, len(s1)):
            for j in range(1, len(s2)):
                acc_cost[i, j] = cost_matrix[i, j] + min(acc_cost[i-1, j], acc_cost[i, j-1], acc_cost[i-1, j-1])
    path = []
    i, j = len(s1) - 1, len(s2) - 1
    path.append((i, j))
    while i > 0 or j > 0:
        if i == 0:
            j -= 1
        elif j == 0:
            i -= 1
        else:
            argmin = np.argmin([acc_cost[i-1, j], acc_cost[i, j-1], acc_cost[i-1, j-1]])
            if argmin == 0: i -= 1
            elif argmin == 1: j -= 1
            else: i -= 1; j -= 1
        path.append((i, j))
    path.reverse()
    alignment_pairs = path
    plt.figure(figsize=(12, 6))
    plt.plot(t, s1, label="Series 1", color=custom_palette[0])
    plt.plot(t, s2, label="Series 2", color=custom_palette[1])
    viz_path = alignment_pairs[::max(1, len(alignment_pairs)//15)]
    for i, j in viz_path:
        plt.plot([t[i], t[j]], [s1[i], s2[j]], 'k--', alpha=0.5)
    highlight_indices = viz_path[1:-1:2]
    for i, j in highlight_indices:
        plt.scatter(t[i], s1[i], color='red', s=50, zorder=5)
        plt.scatter(t[j], s2[j], color='red', s=50, zorder=5)
    if window is not None:
        window_frac = window if isinstance(window, float) and window <= 1 else min(1.0, window / len(s1))
        dtw_dist = dtw_distance(s1, s2, window=window_frac)
        title = f"{title} (Window={window*100:.0f}%)" if isinstance(window, float) and window <= 1 else f"{title} (Window={window} points)"
    else:
        dtw_dist = dtw_distance(s1, s2, window=None)
    plt.text(0.02, 0.95, f"DTW Distance: {dtw_dist:.2f}", transform=plt.gca().transAxes,
             bbox=dict(facecolor='white', alpha=0.8), fontsize=12)
    plt.title(title, fontsize=14)
    plt.xlabel("Time", fontsize=12)
    plt.ylabel("Value", fontsize=12)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def visualize_dtw_matrix(s1, s2, window=None, title="DTW Accumulated Cost Matrix"):
    from scipy.spatial.distance import cdist
    s1_reshaped = s1.reshape(-1, 1)
    s2_reshaped = s2.reshape(-1, 1)
    cost_matrix = cdist(s1_reshaped, s2_reshaped, 'euclidean')
    acc_cost = np.zeros_like(cost_matrix)
    acc_cost[0, 0] = cost_matrix[0, 0]
    for i in range(1, len(s1)):
        acc_cost[i, 0] = acc_cost[i-1, 0] + cost_matrix[i, 0]
    for j in range(1, len(s2)):
        acc_cost[0, j] = acc_cost[0, j-1] + cost_matrix[0, j]
    if window is not None:
        window_size = int(window * len(s1)) if isinstance(window, float) and 0 <= window <= 1 else window
        for i in range(1, len(s1)):
            for j in range(1, len(s2)):
                if abs(i - j) <= window_size:
                    acc_cost[i, j] = cost_matrix[i, j] + min(acc_cost[i-1, j], acc_cost[i, j-1], acc_cost[i-1, j-1])
                else:
                    acc_cost[i, j] = np.inf
    else:
        for i in range(1, len(s1)):
            for j in range(1, len(s2)):
                acc_cost[i, j] = cost_matrix[i, j] + min(acc_cost[i-1, j], acc_cost[i, j-1], acc_cost[i-1, j-1])
    path = []
    i, j = len(s1) - 1, len(s2) - 1
    path.append((i, j))
    while i > 0 or j > 0:
        if i == 0: j -= 1
        elif j == 0: i -= 1
        else:
            argmin = np.argmin([acc_cost[i-1, j], acc_cost[i, j-1], acc_cost[i-1, j-1]])
            if argmin == 0: i -= 1
            elif argmin == 1: j -= 1
            else: i -= 1; j -= 1
        path.append((i, j))
    path.reverse()
    path_y, path_x = zip(*path)
    plt.figure(figsize=(10, 8))
    colors = [(0.0, custom_palette[1]), (1.0, custom_palette[6])]
    cm = LinearSegmentedColormap.from_list('custom_cmap', colors, N=100)
    display_cost = acc_cost.copy()
    if window is not None:
        display_cost[display_cost == np.inf] = np.nanmax(display_cost[display_cost != np.inf])
    plt.imshow(display_cost, origin='lower', cmap=cm, aspect='auto')
    plt.colorbar(label='Accumulated Cost')
    plt.plot(path_x, path_y, color='white', linewidth=2)
    if window is not None:
        window_frac = window if isinstance(window, float) and window <= 1 else min(1.0, window / len(s1))
        dtw_dist = dtw_distance(s1, s2, window=window_frac)
        title = f"{title} (Window={window*100:.0f}%)" if isinstance(window, float) and window <= 1 else f"{title} (Window={window} points)"
    else:
        dtw_dist = dtw_distance(s1, s2, window=None)
    plt.text(0.02, 0.95, f"DTW Distance: {dtw_dist:.2f}", transform=plt.gca().transAxes,
             bbox=dict(facecolor='white', alpha=0.8), fontsize=12)
    plt.title(title, fontsize=14)
    plt.xlabel("Series 2 Index", fontsize=12)
    plt.ylabel("Series 1 Index", fontsize=12)
    plt.tight_layout()
    plt.show()


def compare_distance_measures():
    t, s1, s2, s3, s4, s5 = create_example_series()
    plot_time_series(t, [s1, s2, s3, s4, s5],
                     ["Original", "Phase shifted", "Amplitude variation", "Different pattern", "Noisy signal"],
                     "Sample Time Series for Distance Measure Comparison")
    visualize_euclidean_matching(t, s1, s2)
    visualize_dtw_matching(t, s1, s2)
    visualize_dtw_matrix(s1, s2)
    window_size = int(0.05 * len(s1))
    visualize_dtw_matching(t, s1, s2, window=window_size)
    visualize_dtw_matrix(s1, s2, window=window_size)
    distances_table = []
    for i, s in enumerate([s2, s3, s4, s5]):
        name = ["Phase shifted", "Amplitude variation", "Different pattern", "Noisy signal"][i]
        distances = calculate_distances(s1, s)
        row = {"Series": name}
        row.update(distances)
        distances_table.append(row)
    df = pd.DataFrame(distances_table)
    print("\nDistance Comparison Table:")
    print(df.to_string(index=False))
    df_plot = df.set_index("Series")
    bar_colors = custom_palette[1:1+len(df_plot.columns)]
    ax = df_plot.plot(kind="bar", figsize=(14, 7), color=bar_colors)
    plt.title("Distance Measures Comparison", fontsize=14)
    plt.ylabel("Distance Value", fontsize=12)
    plt.legend(title="Distance Measure")
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    msm_distances = []
    for i, s in enumerate([s2, s3, s4, s5]):
        name = ["Phase shifted", "Amplitude variation", "Different pattern", "Noisy signal"][i]
        msm_dist = msm_distance(s1, s, c=1.0)
        msm_distances.append({"Series": name, "MSM": msm_dist})
    msm_df = pd.DataFrame(msm_distances)
    print("\nMove-Split-Merge Distance Comparison:")
    print(msm_df.to_string(index=False))
    return df


## Summary Table Helper

In [ ]:
def build_summary_table(methods_preds, y_true, timings=None):
    """Build classifier performance table with optional timing column."""
    rows = []
    for name, val in methods_preds.items():
        if isinstance(val, tuple):
            acc, _ = val
            row = {
                "Method": name,
                "Accuracy": f"{acc:.3f}" if isinstance(acc, float) else str(acc),
                "Precision": "—",
                "Recall": "—",
                "F1": "—",
            }
        else:
            y_pred = val
            row = {
                "Method": name,
                "Accuracy": f"{accuracy_score(y_true, y_pred):.3f}",
                "Precision": f"{precision_score(y_true, y_pred, average='weighted'):.2f}",
                "Recall": f"{recall_score(y_true, y_pred, average='weighted'):.2f}",
                "F1": f"{f1_score(y_true, y_pred, average='weighted'):.2f}",
            }
        if timings is not None:
            row["Time (s)"] = f"{timings[name]:.1f}" if name in timings else "—"
        rows.append(row)
    return pd.DataFrame(rows)


## Pseudo-data for Distance Visualisation
*Block: fig-euclidean-pseudo-data*

In [ ]:
t = np.linspace(0, 1, 100)
s1 = np.sin(2 * np.pi * t * 2)
s2 = np.sin(2 * np.pi * (t * 2 + 0.2))
s3 = 0.5 * np.sin(2 * np.pi * t * 2)
s4 = np.cos(2 * np.pi * t * 3)
s5 = np.sin(2 * np.pi * t * 2) + np.random.normal(0, 0.1, len(t))


## fig-euclidean-alignment

In [ ]:
_fig_save_path[0] = 'images/chap14_images/fig_euclidean_alignment.png'
visualize_euclidean_matching(t, s1, s3, title=None)


## fig-dtw-cost-matrix

In [ ]:
_fig_save_path[0] = 'images/chap14_images/fig_dtw_cost_matrix.png'
visualize_dtw_matrix(s1, s3, title=None)


## fig-dtw-alignment

In [ ]:
_fig_save_path[0] = 'images/chap14_images/fig_dtw_alignment.png'
visualize_dtw_matching(t, s1, s3, title=None)


## DTW Distance Calculations
*Block: dtw-distance-calculations*

In [ ]:
distances = calculate_distances(s1, s3)
distances


## Loading Dodgers Data
*Block: loading-dodgers-data*

In [ ]:
df = pd.read_csv('data/chapter14/dodgerloopday_imputed.csv')

train_df = df[df['source'] == 'train']
test_df = df[df['source'] == 'test']

print(df.head(n=3))


## Data Preparation
*Block: data-prep-tsc*

In [ ]:
def prepare_data(df):
    series_list = []
    labels = []

    for series_id, group in df.groupby('series_id'):
        group = group.sort_values('time')
        series = group['value_filled'].values
        label = group['class'].iloc[0]
        series_list.append(series)
        labels.append(label)

    X = np.array(series_list)
    y = np.array(labels)

    X = X.reshape(X.shape[0], 1, X.shape[1])
    return X, y

X_train, y_train = prepare_data(train_df)
X_test, y_test = prepare_data(test_df)


## KNN Distance Comparison
*Block: knn-distance-comparison*

In [ ]:
%%time
from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier
from sklearn.metrics import accuracy_score, classification_report

classifiers = {
    "Euclidean": KNeighborsTimeSeriesClassifier(n_neighbors=1, distance="euclidean"),
    "Manhattan": KNeighborsTimeSeriesClassifier(n_neighbors=1, distance="manhattan"),
    "Minkowski": KNeighborsTimeSeriesClassifier(n_neighbors=1, distance="minkowski"),
    "DTW": KNeighborsTimeSeriesClassifier(n_neighbors=1, distance="dtw"),
    "WDTW": KNeighborsTimeSeriesClassifier(n_neighbors=1, distance="wdtw"),
    "MSM": KNeighborsTimeSeriesClassifier(n_neighbors=1, distance="msm"),
}

knn_preds = {}
for name, clf in classifiers.items():
    print(f"\nTraining {name} classifier...")
    _start = time.time()
    clf.fit(X_train, y_train)
    timings[name] = time.time() - _start
    y_pred = clf.predict(X_test)
    knn_preds[name] = y_pred
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy


## Table 14.1 — KNN Distance Results
*Block: tbl-knn-distance-results*

In [ ]:
tbl1 = build_summary_table(knn_preds, y_test, timings)
print("Table 14.1: KNN classification results across distance metrics")
print(tbl1.to_markdown(index=False))


## fig-knn-distance-accuracy

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(results.keys(), results.values(), color=custom_palette[:len(results)])
plt.ylabel("Accuracy", fontsize=14)
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)

for i, (method, accuracy) in enumerate(results.items()):
    plt.text(i, accuracy + 0.02, f"{accuracy:.4f}", ha='center', fontsize=14)

plt.xticks(fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=14)
plt.tight_layout()
plt.savefig('images/chap14_images/fig_knn_distance_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()


## Elastic Ensemble
*Block: elastic-ensemble*

In [ ]:
%%time
from aeon.classification.distance_based import ElasticEnsemble

ee = ElasticEnsemble(
    distance_measures=["euclidean", "wdtw", "ddtw", "msm", "twe"],
    proportion_of_param_options=0.1,
    proportion_train_in_param_finding=0.3,
    proportion_train_for_test=0.1,
)
_start = time.time()
ee.fit(X_train, y_train)
timings["ElasticEnsemble"] = time.time() - _start
ee_preds = ee.predict(X_test)
results["ElasticEnsemble"] = accuracy_score(y_test, ee_preds)
print(f"ElasticEnsemble accuracy: {results['ElasticEnsemble']:.4f}")
print(f"Training time: {timings['ElasticEnsemble']:.2f}s")


## Proximity Forest — Default
*Block: proximity-forest-default*

In [ ]:
%%time
from aeon.classification.distance_based import ProximityForest

_start = time.time()
forest = ProximityForest(n_trees=20, n_splitters=5, max_depth=10)
forest.fit(X_train, y_train)
timings["ProximityForest"] = time.time() - _start

forest_preds = forest.predict(X_test)
results["ProximityForest"] = accuracy_score(y_test, forest_preds)
print(f"ProximityForest accuracy: {results['ProximityForest']:.4f}")
print(f"Training time: {timings['ProximityForest']:.2f}s")


## Proximity Forest HP Optimisation — SKIPPED
*Block: proximity-forest-hp-optimisation*

Optuna search skipped; using previously found best parameters.

In [ ]:
# Optuna search skipped — using previously found best parameters
best_params = {'n_trees': 104, 'n_splitters': 7, 'max_depth': 13}
print(f"Best parameters: {best_params}")


## Proximity Forest — Tuned
*Block: proximity-forest-tuned + proximity-forest-tuned-code to rerun*

In [ ]:
%%time
_start = time.time()
best_forest = ProximityForest(
    n_trees=best_params['n_trees'],
    n_splitters=best_params['n_splitters'],
    max_depth=best_params['max_depth']
)
best_forest.fit(X_train, y_train)
timings["ProximityForest (tuned)"] = time.time() - _start

pf_tuned_preds = best_forest.predict(X_test)
results["ProximityForest (tuned)"] = accuracy_score(y_test, pf_tuned_preds)
print(f"ProximityForest (tuned) accuracy: {results['ProximityForest (tuned)']:.4f}")
print(f"Training time: {timings['ProximityForest (tuned)']:.2f}s")


## Table 14.2 — Distance Classifier Comparison
*Block: tbl-distance-results*

In [ ]:
distance_preds = {
    "ElasticEnsemble": ee_preds,
    "ProximityForest": forest_preds,
    "ProximityForest (tuned)": pf_tuned_preds,
}
tbl2 = build_summary_table(distance_preds, y_test, timings)
print("\nTable 14.2: Distance classifier comparison")
print(tbl2.to_markdown(index=False))


## fig-distance-classifiers-comparison

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(results.keys(), results.values(), color=custom_palette[:len(results)])
plt.ylabel("Accuracy", fontsize=14)
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)

for i, (method, accuracy) in enumerate(results.items()):
    plt.text(i, accuracy + 0.02, f"{accuracy:.4f}", ha='center', fontsize=14)

plt.xticks(fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=14)
plt.tight_layout()
plt.savefig('images/chap14_images/fig_distance_classifiers_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## fig-distribution-features

In [ ]:
np.random.seed(673)
t = np.linspace(0, 10, 500)
x = np.sin(t) + 0.5 * np.random.randn(len(t))

def visualize_distribution(x):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

    ax1.plot(x, color=custom_palette[1])
    ax1.set_title('Original Time Series')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Value')

    ax2.hist(x, bins=40, color=custom_palette[2], alpha=0.7)
    ax2.axvline(np.mean(x), color=custom_palette[0], linestyle='--',
                linewidth=3, label=f'Mean: {np.mean(x):.2f}')
    ax2.axvline(np.median(x), color=custom_palette[5], linestyle='-.',
                linewidth=3, label=f'Median: {np.median(x):.2f}')

    variance = np.var(x, ddof=1)
    kurtosis = stats.kurtosis(x, fisher=True)
    ax2.text(0.05, 0.9, f'Variance: {variance:.2f}', transform=ax2.transAxes,
             fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
    ax2.text(0.05, 0.78, f'Kurtosis: {kurtosis:.2f}', transform=ax2.transAxes,
             fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

    ax2.set_title('Distribution of Values')
    ax2.set_xlabel('Value')
    ax2.set_ylabel('Frequency')
    ax2.legend()

    plt.tight_layout()
    plt.show()

_fig_save_path[0] = 'images/chap14_images/fig_distribution_features.png'
visualize_distribution(x)


## fig-stationarity-measures

In [ ]:
def statAv(x, window_size):
    n = len(x)
    m = n // window_size
    window_means = [np.mean(x[i*window_size:(i+1)*window_size]) for i in range(m)]
    return np.std(window_means) / np.std(x)

def visualize_stationarity(x, window_size=50):
    n = len(x)
    m = n // window_size

    window_means = np.array([np.mean(x[i*window_size:(i+1)*window_size]) for i in range(m)])
    window_stds = np.array([np.std(x[i*window_size:(i+1)*window_size], ddof=1) for i in range(m)])
    stat_av = np.std(window_means) / np.std(x)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

    ax1.plot(x, color=custom_palette[0], alpha=0.7)

    for i in range(m):
        start = i * window_size
        end = start + window_size
        if end > len(x):
            end = len(x)
        ax1.plot([start, end-1], [window_means[i], window_means[i]],
                 color=custom_palette[3], linewidth=3)
        ax1.axvline(start, color='gray', linestyle='--', alpha=0.3)

    ax1.set_title('Time Series with Window Means')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Value')

    ax2.plot(range(m), window_means, 'o-', color=custom_palette[2], label='Window Means', markersize=12)
    ax2.plot(range(m), window_stds, 'x-', color=custom_palette[4], label='Window StDevs', markersize=12)
    ax2.axhline(np.mean(x), color=custom_palette[6], linestyle='--',
                label=f'Overall Mean: {np.mean(x):.2f}')

    ax2.text(0.05, 0.9, f'StatAv: {stat_av:.2f}', transform=ax2.transAxes,
             fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

    ax2.set_title('Window Statistics')
    ax2.set_xlabel('Window Index')
    ax2.set_ylabel('Statistic Value')
    ax2.legend()

    plt.tight_layout()
    plt.show()

t = np.linspace(0, 10, 500)
non_stationary = np.sin(t) + 0.2 * t + 0.3 * np.random.randn(len(t))

_fig_save_path[0] = 'images/chap14_images/fig_stationarity_measures.png'
visualize_stationarity(non_stationary)


## fig-autocorrelation-features

In [ ]:
def autocorrelation(x, max_lag=50):
    n = len(x)
    mean_x = np.mean(x)
    var_x = np.var(x, ddof=1)

    ac = np.zeros(max_lag + 1)
    ac[0] = 1

    for lag in range(1, max_lag + 1):
        for t in range(n - lag):
            ac[lag] += (x[t] - mean_x) * (x[t + lag] - mean_x)
        ac[lag] /= (n - lag) * var_x

    return ac

def visualize_autocorrelation(x, max_lag=50):
    ac = autocorrelation(x, max_lag)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

    ax1.plot(x, color=custom_palette[1])
    ax1.set_title('Original Time Series')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Value')

    ax2.stem(range(max_lag + 1), ac, linefmt=custom_palette[2],
             markerfmt=f'{custom_palette[2]}', basefmt='gray')

    ci = 1.96 / np.sqrt(len(x))
    ax2.axhline(ci, color=custom_palette[6], linestyle='--', alpha=0.7)
    ax2.axhline(-ci, color=custom_palette[6], linestyle='--', alpha=0.7)
    ax2.fill_between(range(max_lag + 1), -ci, ci, color='gray', alpha=0.2)

    ax2.set_title('Autocorrelation Function')
    ax2.set_xlabel('Lag')
    ax2.set_ylabel('Autocorrelation')

    plt.tight_layout()
    plt.show()

t = np.linspace(0, 5, 500)
seasonal = 2 * np.sin(2 * np.pi * t) + np.random.randn(len(t)) * 0.5

_fig_save_path[0] = 'images/chap14_images/fig_autocorrelation_features.png'
visualize_autocorrelation(seasonal)


## fig-high-fluctuation

In [ ]:
blue = '#0173B2'
black = '#000000'

plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['font.size'] = 12

def create_spiky_series(n_points=1000, spike_interval=200, noise_level=0.1):
    time_arr = np.arange(n_points)
    signal = np.zeros(n_points)
    for i in range(50, n_points, spike_interval):
        if i < n_points:
            signal[i-5:i+5] = np.exp(-np.abs(np.arange(-5, 5)) / 2)
            signal[i] = 2
    signal += np.random.normal(0, noise_level, n_points)
    return time_arr, signal

def calc_incremental_differences(signal):
    return np.diff(signal)

def calc_high_fluctuation(signal, threshold_factor=0.04):
    diffs = np.abs(calc_incremental_differences(signal))
    threshold = threshold_factor * np.std(signal)
    high_fluc = np.sum(diffs > threshold) / len(diffs)
    return high_fluc, diffs, threshold

def plot_high_fluctuation_example():
    time_arr, signal = create_spiky_series()
    high_fluc, diffs, threshold = calc_high_fluctuation(signal)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [2, 1]})

    ax1.plot(time_arr, signal, color=black, linewidth=1.5)
    ax1.set_title(f'Spiky Time Series [high_fluctuation = {high_fluc:.3f}]')
    ax1.set_ylabel('Value')
    ax1.set_xlabel('Time')

    bins = np.linspace(0, 2, 50)
    ax2.hist(np.abs(diffs)/np.std(signal), bins=bins, color=blue, alpha=0.7,
             edgecolor='white', linewidth=0.5)
    ax2.axvline(threshold/np.std(signal), color='red', linestyle='--',
                linewidth=1.5, label=f'Threshold (0.04σ)')
    ax2.set_xlabel('Incremental differences (abs sigma)')
    ax2.set_ylabel('Frequency')
    ax2.legend()

    plt.tight_layout()
    return fig

fig1 = plot_high_fluctuation_example()
plt.savefig('images/chap14_images/fig_high_fluctuation.png', dpi=150, bbox_inches='tight')
plt.show()


## fig-whitening-timescale

In [ ]:
def create_oscillating_series(n_points=400):
    time_arr = np.arange(n_points)
    frequency = 0.15 + 0.05 * np.sin(time_arr * 0.01)
    base_signal = 10 * np.sin(time_arr * frequency)
    trend = 3 * np.sin(time_arr * 0.02)
    noise = np.zeros(n_points)
    noise[0] = np.random.normal(0, 1)
    for i in range(1, n_points):
        noise[i] = 0.7 * noise[i-1] + np.random.normal(0, 1.5)
    weight = 0.5 + 0.5 * (time_arr > 200)
    signal = base_signal + trend + weight * noise
    return time_arr, signal

def calc_autocorrelation(signal, max_lag=20):
    n = len(signal)
    mean = np.mean(signal)
    var = np.var(signal)
    acf = np.zeros(max_lag + 1)
    acf[0] = 1
    for lag in range(1, max_lag + 1):
        numerator = np.sum((signal[lag:] - mean) * (signal[:n-lag] - mean))
        acf[lag] = numerator / ((n - lag) * var)
    return acf

def find_first_zero_crossing(acf):
    for i in range(1, len(acf)):
        if acf[i] <= 0:
            if i > 0:
                x0, y0 = i-1, acf[i-1]
                x1, y1 = i, acf[i]
                zero_cross = x0 + (0 - y0) * (x1 - x0) / (y1 - y0)
                return zero_cross
            return i
    return len(acf)

def calc_whiten_timescale(signal, max_lag=20):
    acf_orig = calc_autocorrelation(signal, max_lag)
    tau_orig = find_first_zero_crossing(acf_orig)
    diffs = calc_incremental_differences(signal)
    acf_diffs = calc_autocorrelation(diffs, max_lag)
    tau_diffs = find_first_zero_crossing(acf_diffs)
    ratio = tau_diffs / tau_orig if tau_orig > 0 else np.nan
    return ratio, acf_orig, acf_diffs, tau_orig, tau_diffs

def plot_whiten_timescale_example():
    time_arr, signal = create_oscillating_series()
    diffs = calc_incremental_differences(signal)
    time_diffs = time_arr[1:]
    ratio, acf_orig, acf_diffs, tau_orig, tau_diffs = calc_whiten_timescale(signal)
    lags = np.arange(len(acf_orig))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [1, 1]})

    ax1.plot(time_arr, signal, color=black, linewidth=1.5, label='Original')
    ax1.plot(time_diffs, diffs, color=blue, linewidth=1.5, alpha=0.7, linestyle='--', label='Differences')
    ax1.set_title(f'Oscillating Time Series [whiten_timescale = {ratio:.3f}]')
    ax1.set_ylabel('Value')
    ax1.set_xlabel('Time')
    ax1.legend()

    ax2.plot(lags, acf_orig, color=black, linewidth=1.5, marker='o', markersize=5, label='Original ACF')
    ax2.plot(lags, acf_diffs, color=blue, linewidth=1.5, marker='^', markersize=5, label='Differences ACF')
    ax2.axhline(0, color='gray', linestyle=':', linewidth=1.5)
    ax2.plot(tau_orig, 0, 'ro', markersize=14, label=f'First Zero-Crossing (orig): {tau_orig:.1f}')
    ax2.plot(tau_diffs, 0, 'go', markersize=14, label=f'First Zero-Crossing (diff): {tau_diffs:.1f}')
    ax2.set_title(f'Autocorrelation Functions')
    ax2.set_ylabel('Autocorrelation')
    ax2.set_xlabel('Time lag, τ')
    ax2.set_ylim(-0.7, 1.1)
    ax2.legend()

    plt.tight_layout()
    return fig

fig2 = plot_whiten_timescale_example()
plt.savefig('images/chap14_images/fig_whitening_timescale.png', dpi=150, bbox_inches='tight')
plt.show()


## fig-forecast-error

In [ ]:
from scipy import stats

blue = '#0173B2'
black = '#000000'

plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['font.size'] = 16

np.random.seed(59028)
t = np.arange(100)

predictable = np.sin(t[:50] * 0.3) + np.random.normal(0, 0.2, 50)
less_predictable = np.sin(t[50:] * 0.3) + np.random.normal(0, 0.8, 50)

time_series = np.concatenate([predictable, less_predictable])
time_series = stats.zscore(time_series)

forecasts = np.zeros_like(time_series)
forecasts[:3] = np.nan

for i in range(3, len(time_series)):
    forecasts[i] = np.mean(time_series[i-3:i])

residuals = time_series[3:] - forecasts[3:]

forecast_error_full = np.std(residuals)
forecast_error_region1 = np.std(residuals[:47])
forecast_error_region2 = np.std(residuals[47:])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(t, time_series, color=black, linewidth=1.5, label='Original Time Series')
ax1.plot(t[3:], forecasts[3:], color=blue, linewidth=1.5, label='3-Point Rolling Mean Forecast')
ax1.set_xlabel('Time', fontsize=16)
ax1.set_ylabel('Value (z-scored)', fontsize=16)
ax1.legend(fontsize=16)

ax1.axvline(x=50, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
ax1.text(30, max(time_series)+0.3, 'Lower Noise Region', fontsize=16, ha='center')
ax1.text(75, max(time_series)+0.3, 'Higher Noise Region', fontsize=16, ha='center')

ax2.plot(t[3:], residuals, color=blue, linewidth=1.5)
ax2.axhline(y=0, color=black, linestyle='-', alpha=0.3, linewidth=1.5)
ax2.fill_between(t[3:], 0, residuals, color=blue, alpha=0.2)
ax2.set_xlabel('Time', fontsize=16)
ax2.set_ylabel('Forecast Error', fontsize=16)
ax2.set_title(f'Forecast Residuals', fontsize=18)

ax2.text(25, max(residuals)-0.2, f'Region 1 Error = {forecast_error_region1:.3f}',
         fontsize=16, ha='center', bbox=dict(facecolor='white', alpha=0.8))
ax2.text(75, max(residuals)-0.2, f'Region 2 Error = {forecast_error_region2:.3f}',
         fontsize=16, ha='center', bbox=dict(facecolor='white', alpha=0.8))
ax2.text(50, min(residuals)+0.3, f'Full Series Error = {forecast_error_full:.3f}',
         fontsize=16, ha='center', bbox=dict(facecolor='white', alpha=0.8))

ax2.axhline(y=1, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
ax2.axhline(y=-1, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
ax2.text(103, 1.1, 'σ = 1', color='red', fontsize=16, ha='right')

plt.tight_layout()
plt.savefig('images/chap14_images/fig_forecast_error.png', dpi=150, bbox_inches='tight')
plt.show()


## fig-entropy-features

In [ ]:
def sample_entropy(x, m=2, r=0.2):
    n = len(x)
    r = r * np.std(x, ddof=1)

    def count_matches(template, patterns, r):
        return sum(1 for pattern in patterns if max(abs(template - pattern)) <= r)

    def create_vectors(m):
        return np.array([x[i:i+m] for i in range(n-m+1)])

    vectors_m = create_vectors(m)
    vectors_m1 = create_vectors(m+1)

    B = 0
    A = 0

    for i in range(n-m):
        template_m = vectors_m[i]
        template_m1 = vectors_m1[i]

        similar_m = count_matches(template_m,
                                  [vectors_m[j] for j in range(n-m+1) if abs(j-i) > 0], r)
        similar_m1 = count_matches(template_m1,
                                   [vectors_m1[j] for j in range(n-m) if abs(j-i) > 0], r)

        B += similar_m / (n-m-1)
        A += similar_m1 / (n-m-1)

    B /= (n-m)
    A /= (n-m)

    if A == 0 or B == 0:
        return float('inf')
    return -np.log(A/B)

def visualize_entropy(x, embedding_dims=range(1, 5)):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

    ax1.plot(x, color=custom_palette[1])
    ax1.set_title('Original Time Series')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Value')

    entropies = []
    for m in embedding_dims:
        se = sample_entropy(x, m=m)
        entropies.append(se)

    ax2.plot(embedding_dims, entropies, 'o-', color=custom_palette[2], linewidth=2, markersize=10)
    ax2.set_title('Sample Entropy for Different Embedding Dimensions')
    ax2.set_xlabel('Embedding Dimension (m)')
    ax2.set_ylabel('Sample Entropy')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

np.random.seed(78447)
n = 200

random_noise = np.random.randn(n)

t = np.linspace(0, 4*np.pi, n)
sine_wave = np.sin(t)

chaotic = np.zeros(n)
chaotic[0] = 0.5
r = 3.9
for i in range(1, n):
    chaotic[i] = r * chaotic[i-1] * (1 - chaotic[i-1])

_fig_save_path[0] = 'images/chap14_images/fig_entropy_features.png'
visualize_entropy(chaotic)


## fig-fourier-features

In [ ]:
def visualize_fourier(x, sampling_rate=1.0):
    n = len(x)

    fft_vals = np.fft.fft(x) / np.sqrt(n)
    fft_freq = np.fft.fftfreq(n, d=1/sampling_rate)

    pos_freq_idx = fft_freq > 0
    freqs = fft_freq[pos_freq_idx]
    magnitudes = np.abs(fft_vals[pos_freq_idx])

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

    ax1.plot(np.arange(n)/sampling_rate, x, color=custom_palette[1])
    ax1.set_title('Original Time Series')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Value')

    ax2.stem(freqs, magnitudes, linefmt=custom_palette[2],
             markerfmt=f'{custom_palette[2]}', basefmt='gray')
    ax2.set_title('Frequency Spectrum (Fourier Transform)')
    ax2.set_xlabel('Frequency')
    ax2.set_ylabel('Magnitude')
    ax2.set_xlim(0, sampling_rate/2)

    top_k = 3
    dominant_idx = np.argsort(magnitudes)[-top_k:]
    for i, idx in enumerate(dominant_idx):
        freq = freqs[idx]
        mag = magnitudes[idx]
        ax2.annotate(f'{freq:.2f} Hz',
                    xy=(freq, mag),
                    xytext=(freq + 3, mag + 0.5),
                    color=custom_palette[i+4],
                    arrowprops=dict(facecolor=custom_palette[i+4], shrink=0.05),
                    horizontalalignment='left')

    plt.tight_layout()
    plt.show()

t = np.linspace(0, 2, 500)
multi_freq = 3*np.sin(2*np.pi*2*t) + 1.5*np.sin(2*np.pi*5*t) + 0.5*np.sin(2*np.pi*10*t) + 0.5*np.random.randn(len(t))

_fig_save_path[0] = 'images/chap14_images/fig_fourier_features.png'
visualize_fourier(multi_freq, sampling_rate=250)


## fig-dfa

In [ ]:
def dfa(x, scales=None):
    if scales is None:
        scales = np.logspace(1, 2.6, 20).astype(int)
        scales = np.unique(scales)

    y = np.cumsum(x - np.mean(x))

    fluctuations = []
    for scale in scales:
        n_segments = len(y) // scale
        fluct = 0

        for i in range(n_segments):
            segment = y[i*scale:(i+1)*scale]
            time_index = np.arange(len(segment))
            coeffs = np.polyfit(time_index, segment, 1)
            trend = np.polyval(coeffs, time_index)
            fluct += np.sum((segment - trend)**2)

        fluctuations.append(np.sqrt(fluct / (n_segments * scale)))

    coeffs = np.polyfit(np.log(scales), np.log(fluctuations), 1)
    alpha = coeffs[0]

    return scales, fluctuations, alpha

def visualize_dfa(x):
    scales, fluctuations, alpha = dfa(x)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

    y = np.cumsum(x - np.mean(x))
    ax1.plot(y, color=custom_palette[1])

    scale = scales[len(scales)//2]
    segment_start = 0
    segment = y[segment_start:segment_start+scale]
    time_index = np.arange(len(segment))
    coeffs = np.polyfit(time_index, segment, 1)
    trend = np.polyval(coeffs, time_index)

    ax1.plot(range(segment_start, segment_start+scale), segment, color=custom_palette[2], linewidth=2)
    ax1.plot(range(segment_start, segment_start+scale), trend, color=custom_palette[6], linewidth=2,
             linestyle='--')

    ax1.set_title('Integrated Time Series with Sample Detrending')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Integrated Value')

    ax2.loglog(scales, fluctuations, 'o', color=custom_palette[3])

    log_scales = np.log(scales)
    log_fluct = np.log(fluctuations)
    fit_line = np.exp(np.polyval([alpha, log_fluct[0] - alpha * log_scales[0]], np.log(scales)))
    ax2.loglog(scales, fit_line, color=custom_palette[5], linewidth=2)

    ax2.text(0.05, 0.9, f'Scaling Exponent α = {alpha:.3f}', transform=ax2.transAxes,
             fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

    ax2.set_title('Detrended Fluctuation Analysis')
    ax2.set_xlabel('Scale (log)')
    ax2.set_ylabel('Fluctuation (log)')

    plt.tight_layout()
    plt.show()

np.random.seed(42)
n = 2000

white_noise = np.random.randn(n)
brownian = np.cumsum(np.random.randn(n))

def fgn(n, H=0.8):
    t = np.arange(n+1)
    fbm = np.zeros(n+1)
    for i in range(1, n+1):
        fbm[i] = fbm[i-1] + np.random.randn() * (i**H - (i-1)**H)
    return np.diff(fbm)

long_range_correlated = fgn(n, H=0.8)

_fig_save_path[0] = 'images/chap14_images/fig_dfa.png'
visualize_dfa(long_range_correlated)


## fig-outlier-timing

In [ ]:
blue = '#0173B2'
black = '#000000'
red = '#D55E00'
green = '#009E73'

plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['font.size'] = 16

def calculate_outlier_timing(x, direction='pos', threshold_pct=0.01):
    x_z = stats.zscore(x)
    n = len(x_z)

    if direction == 'pos':
        max_val = np.max(x_z)
        thresholds = np.linspace(0, max_val, int(1/threshold_pct))
    else:
        min_val = np.min(x_z)
        thresholds = np.linspace(0, min_val, int(1/threshold_pct))

    rmds = []
    for threshold in thresholds:
        if direction == 'pos':
            over_threshold_indices = np.where(x_z > threshold)[0]
        else:
            over_threshold_indices = np.where(x_z < threshold)[0]

        if len(over_threshold_indices) > 0:
            median_idx = np.median(over_threshold_indices)
            rmd = 2 * (median_idx / (n - 1)) - 1
            rmds.append(rmd)

    metric = np.median(rmds) if rmds else np.nan
    return metric, thresholds[:len(rmds)], np.array(rmds)

def generate_time_series_with_outliers(n=400, outlier_position='start', n_outliers=20, magnitude=5):
    x = np.zeros(n)
    x[0] = np.random.randn()
    for i in range(1, n):
        x[i] = 0.7 * x[i-1] + 0.3 * np.random.randn()

    if outlier_position == 'start':
        outlier_indices = np.random.choice(np.arange(n//5), n_outliers, replace=False)
    elif outlier_position == 'middle':
        outlier_indices = np.random.choice(np.arange(2*n//5, 3*n//5), n_outliers, replace=False)
    elif outlier_position == 'end':
        outlier_indices = np.random.choice(np.arange(4*n//5, n), n_outliers, replace=False)
    else:
        outlier_indices = np.random.choice(np.arange(n), n_outliers, replace=False)

    outlier_signs = np.random.choice([-1, 1], n_outliers)
    for i, idx in enumerate(outlier_indices):
        x[idx] += outlier_signs[i] * magnitude

    return x, outlier_indices

def visualize_outlier_timing(x, title="Time Series with Outliers"):
    pos_metric, pos_thresholds, pos_rmds = calculate_outlier_timing(x, 'pos')
    neg_metric, neg_thresholds, neg_rmds = calculate_outlier_timing(x, 'neg')

    x_z = stats.zscore(x)
    n = len(x_z)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    ax1.plot(x_z, color=black, linewidth=1.5)

    pos_threshold = np.percentile(x_z, 95)
    pos_outlier_indices = np.where(x_z > pos_threshold)[0]
    ax1.scatter(pos_outlier_indices, x_z[pos_outlier_indices], color=red, s=100, zorder=10,
               label='Positive Outliers')

    neg_threshold = np.percentile(x_z, 5)
    neg_outlier_indices = np.where(x_z < neg_threshold)[0]
    ax1.scatter(neg_outlier_indices, x_z[neg_outlier_indices], color=green, s=100, zorder=10,
               label='Negative Outliers')

    ax1.axhline(y=pos_threshold, color=red, linestyle='--', alpha=0.7, linewidth=1.5)
    ax1.axhline(y=neg_threshold, color=green, linestyle='--', alpha=0.7, linewidth=1.5)
    ax1.set_xlabel('Time', fontsize=16)
    ax1.set_ylabel('Value (z-scored)', fontsize=16)
    ax1.set_title(title, fontsize=18)
    ax1.legend(fontsize=16)

    ax2.plot(pos_thresholds, pos_rmds, 'o-', color=red, label='Positive Outliers')
    ax2.plot(neg_thresholds, neg_rmds, 'o-', color=green, label='Negative Outliers')
    ax2.axhline(y=0, color=black, linestyle='-', alpha=0.3, linewidth=1.5)
    ax2.axhline(y=-1, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)
    ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)

    ax2.text(0.05, 0.92, f'Positive Outlier Timing = {pos_metric:.3f}', transform=ax2.transAxes,
            fontsize=14, color=red, bbox=dict(facecolor='white', alpha=0.8))
    ax2.text(0.05, 0.75, f'Negative Outlier Timing = {neg_metric:.3f}', transform=ax2.transAxes,
            fontsize=14, color=green, bbox=dict(facecolor='white', alpha=0.8))

    ax2.set_xlabel('Threshold Value', fontsize=16)
    ax2.set_ylabel('Rescaled Median Position', fontsize=16)
    ax2.set_title('Outlier Timing Analysis', fontsize=18)
    ax2.set_ylim(-1.1, 1.1)
    ax2.legend(fontsize=16)

    if pos_metric < -0.3:
        pos_interpretation = "Positive outliers occur mostly near the start"
    elif pos_metric > 0.3:
        pos_interpretation = "Positive outliers occur mostly near the end"
    else:
        pos_interpretation = "Positive outliers are distributed throughout"

    if neg_metric < -0.3:
        neg_interpretation = "Negative outliers occur mostly near the start"
    elif neg_metric > 0.3:
        neg_interpretation = "Negative outliers occur mostly near the end"
    else:
        neg_interpretation = "Negative outliers are distributed throughout"

    fig.text(0.5, 0.01, f"{pos_interpretation}\n{neg_interpretation}",
             ha='center', fontsize=14, bbox=dict(facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.show()

    return pos_metric, neg_metric

np.random.seed(42)

ts_end, _ = generate_time_series_with_outliers(outlier_position='end')
_fig_save_path[0] = 'images/chap14_images/fig_outlier_timing.png'
pos_metric_end, neg_metric_end = visualize_outlier_timing(ts_end, "Time Series with Outliers at End")


## fig-shapelet-features (Shapelet Illustration)

In [ ]:
from scipy.spatial.distance import euclidean

plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['font.size'] = 18

teal = '#0173B2'
red = '#D55E00'

def generate_time_series(n_points=100, pattern_type='class1', noise_level=0.2):
    np.random.seed(42)
    t = np.linspace(0, 10, n_points)
    noise = np.random.normal(0, noise_level, n_points)
    base = 0.5 * np.sin(t) + noise
    if pattern_type == 'class1':
        pattern_positions = [10, 30, 50, 70, 90]
        for pos in pattern_positions:
            if pos < n_points - 10:
                base[pos:pos+10] += 0.8 * np.sin(np.linspace(0, np.pi, 10))
    else:
        pattern_positions = [10, 25, 40, 55, 70, 85]
        for pos in pattern_positions:
            if pos < n_points - 6:
                zigzag = np.array([0, 0.7, 0, 0.7, 0, 0.7])
                base[pos:pos+6] += zigzag
    return base

def shapelet_distance(shapelet, time_series):
    shapelet_len = len(shapelet)
    min_dist = float('inf')
    min_position = 0
    for i in range(len(time_series) - shapelet_len + 1):
        subsequence = time_series[i:i+shapelet_len]
        dist = euclidean(shapelet, subsequence)
        if dist < min_dist:
            min_dist = dist
            min_position = i
    return min_dist, min_position

n_points = 120
ts_class1 = generate_time_series(n_points, 'class1')
ts_class2 = generate_time_series(n_points, 'class2')

shapelet_len = 10
shapelet = 0.8 * np.sin(np.linspace(0, np.pi, shapelet_len))

dist1, pos1 = shapelet_distance(shapelet, ts_class1)
dist2, pos2 = shapelet_distance(shapelet, ts_class2)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12))

ax1.plot(shapelet, 'o-', color=red, linewidth=2, markersize=6)
ax1.set_title('Shapelet', fontsize=18)
ax1.set_ylabel('Value', fontsize=18)
ax1.set_xticklabels([])

ax2.plot(ts_class1, color=teal, linewidth=1.5)
ax2.set_title('Class 1 - Sliding Window Matching', fontsize=18)
ax2.set_ylabel('Value', fontsize=18)
ax2.set_xticklabels([])

slide_positions = [10, 30, 50, 70]
distances = []

for i, pos in enumerate(slide_positions):
    if pos < n_points - shapelet_len:
        subsequence = np.arange(pos, pos+shapelet_len)
        dist = euclidean(shapelet, ts_class1[subsequence])
        distances.append(dist)
        alpha = 1.0 if pos == pos1 else 0.5
        ax2.plot(subsequence, ts_class1[subsequence], 'o-', color=red,
                 linewidth=2, markersize=6, alpha=alpha)
        ax2.text(pos-3, min(ts_class1)-0.2-i*0.1, f'Distance: {dist:.3f}',
                fontsize=16, color=red if pos == pos1 else 'darkred')

subsequence1 = np.arange(pos1, pos1+shapelet_len)
ax2.plot(subsequence1, ts_class1[subsequence1], 'o-', color='darkred',
         linewidth=3, markersize=8)
ax2.text(pos1+shapelet_len/2, min(ts_class1)-0.6, 'Best Match',
        fontsize=16, color='darkred', ha='center',
        bbox=dict(facecolor='white', alpha=0.8))

slide_range = range(n_points - shapelet_len + 1)
all_distances = []

for i in slide_range:
    subsequence = np.arange(i, i+shapelet_len)
    dist = euclidean(shapelet, ts_class1[subsequence])
    all_distances.append(dist)

ax3.plot(all_distances, color=teal, linewidth=1.5)
ax3.set_title('Distance to Shapelet at Each Position', fontsize=18)
ax3.set_xlabel('Starting Position of Subsequence', fontsize=18)
ax3.set_ylabel('Euclidean Distance', fontsize=18)

ax3.scatter(pos1, dist1, color=red, s=100)
ax3.annotate(f'Minimum Distance: {dist1:.3f}', xy=(pos1, dist1),
             xytext=(pos1+10, dist1+0.5), fontsize=16, color=red,
             arrowprops=dict(facecolor=red, shrink=0.05, width=1.5))

plt.tight_layout()
plt.savefig('images/chap14_images/fig_shapelet_features.png', dpi=150, bbox_inches='tight')
plt.show()


## Featurisation Example — Catch22 Transform
*Block: featurisation-example*

In [ ]:
from aeon.transformations.collection.feature_based import Catch22

feature_transformer = Catch22(features="all", n_jobs=10, replace_nans=True)
series_transformed = feature_transformer.fit_transform(X_train)


## Featurisation Table
*Block: featurisation-table*

In [ ]:
from IPython.display import Markdown

feature_names = [
    "DN_HistogramMode_5", "DN_HistogramMode_10",
    "CO_f1ecac", "CO_FirstMin_ac",
    "SB_BinaryStats_mean_longstretch1",
    "DN_OutlierInclude_p_001_mdrmd", "DN_OutlierInclude_n_001_mdrmd",
    "FC_LocalSimple_mean1_tauresrat",
    "CO_trev_1_num",
    "CO_HistogramAMI_even_2_5",
    "IN_AutoMutualInfoStats_40_gaussian_fmmi",
    "MD_hrv_classic_pnn40",
    "SB_BinaryStats_diff_longstretch0",
    "SB_TransitionMatrix_3ac_sumdiagcov",
    "PD_PeriodicityWang_th0_01",
    "CO_Embed2_Dist_tau_d_expfit_meandiff",
    "IN_AutoMutualInfoStats_40_gaussian_std",
    "FC_LocalSimple_mean3_stderr",
    "CO_HistogramAMI_even_10_5",
    "CO_Embed2_Dist_tau_d_expfit_lambda",
    "MD_hrv_classic_pnn20",
    "SB_BinaryStats_mean_longstretch0"
]

featurised_df = pd.DataFrame(series_transformed, columns=feature_names)
featurised_df.insert(0, 'series_id', range(len(series_transformed)))
featurised_df.insert(1, 'label', y_train)

Markdown(featurised_df.head(5).to_markdown(index=False))


## catch22 Classification
*Block: catch22-classification*

In [ ]:
%%time
from aeon.classification.feature_based import Catch22Classifier
from sklearn.ensemble import RandomForestClassifier

c22 = Catch22Classifier(
    estimator=RandomForestClassifier(
        n_estimators=291,
        max_depth=5,
        min_samples_split=12,
        min_samples_leaf=5,
        max_features='sqrt',
        bootstrap=True,
        class_weight='balanced',
        random_state=3542
    ),
    features='all',
    catch24=False,
    outlier_norm=True,
    replace_nans=True,
    n_jobs=10,
    random_state=3542
)
_start = time.time()
c22.fit(X_train, y_train)
timings["catch22class"] = time.time() - _start
c22_preds = c22.predict(X_test)
results["catch22class"] = accuracy_score(y_test, c22_preds)
print(f"catch22 accuracy: {results['catch22class']:.4f}, time: {timings['catch22class']:.2f}s")


## FreshPRINCE — Timeout Wrapper
*Block: freshprince-classification-efficient*

On Windows, `signal.SIGALRM` is unavailable. Set `ATTEMPT_FRESHPRINCE = True` below if you want to try it (may take >1 h).

In [ ]:
import platform

ATTEMPT_FRESHPRINCE = False   # Set True to attempt; may not finish on Windows

freshprince_completed = False

if ATTEMPT_FRESHPRINCE:
    from aeon.classification.feature_based import FreshPRINCEClassifier

    FRESHPRINCE_TIMEOUT = 600  # 10 minutes

    if platform.system() != 'Windows':
        import signal

        def _fp_timeout_handler(signum, frame):
            raise TimeoutError("FreshPRINCE timed out")

        signal.signal(signal.SIGALRM, _fp_timeout_handler)
        signal.alarm(FRESHPRINCE_TIMEOUT)

    try:
        fp = FreshPRINCEClassifier(
            default_fc_parameters="efficient",
            n_jobs=10,
            n_estimators=200,
            verbose=-1,
            random_state=866
        )
        _start = time.time()
        fp.fit(X_train, y_train)
        timings["FreshPRINCE"] = time.time() - _start
        fp_preds = fp.predict(X_test)
        results["freshprince"] = accuracy_score(y_test, fp_preds)
        freshprince_completed = True
        print(f"FreshPRINCE accuracy: {results['freshprince']:.4f}")
    except (Exception,) as e:
        print(f"FreshPRINCE did not complete: {e}")
    finally:
        if platform.system() != 'Windows':
            signal.alarm(0)
else:
    print("FreshPRINCE skipped (ATTEMPT_FRESHPRINCE=False). "
          "Set to True and re-run if you wish to attempt it.")


## Table 14.3 — Feature-based Classifier Comparison
*Block: tbl-feature-results*

In [ ]:
feature_preds = {"catch22": c22_preds}
if freshprince_completed:
    feature_preds["FreshPRINCE"] = fp_preds
else:
    feature_preds["FreshPRINCE"] = ("DNF", None)

tbl3 = build_summary_table(feature_preds, y_test, timings)
print("\nTable 14.3: Feature-based classifier comparison")
print(tbl3.to_markdown(index=False))


## fig-feature-classifiers-comparison

In [ ]:
selected_keys = ['Euclidean', 'WDTW', 'ProximityForest (tuned)', 'catch22class']
results_filtered = {k: results[k] for k in selected_keys if k in results}

plt.figure(figsize=(10, 6))
plt.bar(results_filtered.keys(), results_filtered.values(), color=custom_palette[:len(results_filtered)])
plt.ylabel("Accuracy", fontsize=18)
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)

for i, (method, accuracy) in enumerate(results_filtered.items()):
    plt.text(i, accuracy + 0.02, f"{accuracy:.4f}", ha='center', fontsize=18)

plt.xticks(fontsize=18)
plt.xticks(rotation=45, ha='right', fontsize=16)
plt.yticks(fontsize=18)
plt.tight_layout()
plt.savefig('images/chap14_images/fig_feature_classifiers_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## Shapelet Classification
*Block: Shaplet classification*

In [ ]:
%%time
try:
    from aeon.classification.shapelet_based import (
        ShapeletTransformClassifier,
        RDSTClassifier,
        SASTClassifier,
        LearningShapeletClassifier
    )
except ImportError:
    from aeon.classification.shapelet_based import (
        ShapeletTransformClassifier,
        RDSTClassifier,
        SASTClassifier,
    )
    LearningShapeletClassifier = None

RANDOM_STATE = 3542

rf_params = {
    'n_estimators': 300,
    'max_depth': 5,
    'min_samples_split': 10,
    'class_weight': 'balanced',
    'random_state': RANDOM_STATE
}

classifiers = {
    "ShapeletTransform": ShapeletTransformClassifier(
        estimator=RandomForestClassifier(**rf_params),
        max_shapelets=10,
        n_shapelet_samples=500,
        random_state=RANDOM_STATE
    ),
    "RDST": RDSTClassifier(
        estimator=RandomForestClassifier(**rf_params),
        random_state=RANDOM_STATE
    ),
    "SAST": SASTClassifier(
        classifier=RandomForestClassifier(**rf_params),
    )
}

shapelet_preds = {}
for name, clf in classifiers.items():
    _start = time.time()
    clf.fit(X_train, y_train)
    timings[name] = time.time() - _start
    preds = clf.predict(X_test)
    shapelet_preds[name] = preds
    results[name] = accuracy_score(y_test, preds)
    print(f"{name}: accuracy={results[name]:.4f}, time={timings[name]:.2f}s")


## Table 14.4 — Shapelet Classifier Comparison
*Block: tbl-shapelet-results*

In [ ]:
tbl4 = build_summary_table(shapelet_preds, y_test, timings)
print("\nTable 14.4: Shapelet classifier comparison")
print(tbl4.to_markdown(index=False))


## fig-shapelet-classifiers-comparison

In [ ]:
selected_keys = ['Euclidean', 'WDTW', 'ProximityForest (tuned)', 'catch22class',
                 'ShapeletTransform', 'RDST', 'SAST']
results_filtered = {k: results[k] for k in selected_keys if k in results}

plt.figure(figsize=(10, 6))
plt.bar(results_filtered.keys(), results_filtered.values(), color=custom_palette[:len(results_filtered)])
plt.ylabel("Accuracy", fontsize=18)
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)

for i, (method, accuracy) in enumerate(results_filtered.items()):
    plt.text(i, accuracy + 0.02, f"{accuracy:.4f}", ha='center', fontsize=18)

plt.xticks(fontsize=18)
plt.xticks(rotation=45, ha='right', fontsize=16)
plt.yticks(fontsize=18)
plt.tight_layout()
plt.savefig('images/chap14_images/fig_shapelet_classifiers_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## fig-dictionary-approach (Dictionary Visualisation)

In [ ]:
from scipy import signal

np.random.seed(2542)

t = np.linspace(0, 1000, 1000)
sample = signal.chirp(t, f0=0.01, f1=0.05, t1=500, method='quadratic') * 0.5
envelope = np.exp(-0.5 * ((t - 500) / 150) ** 2)
sample = sample * envelope
sample += np.random.normal(0, 0.02, size=len(sample))

window_size = 100
window_positions = [50, 200, 400, 600, 800]
windows = []
for pos in window_positions:
    if pos + window_size <= len(sample):
        windows.append(sample[pos:pos+window_size])

normalized_windows = []
for window in windows:
    mean = np.mean(window)
    std = np.std(window)
    if std > 0:
        normalized_windows.append((window - mean) / std)
    else:
        normalized_windows.append(window - mean)

patterns = ['bca', 'cab', 'bbb', 'bbc', 'abc', 'aaa', 'acc', 'bcb', 'ccc', 'cba']
pattern_counts = [3.2, 2.8, 2.5, 2.2, 2.1, 1.7, 1.4, 1.2, 1.2, 0.5]

plt.figure(figsize=(10, 10))
plt.rcParams.update({'font.size': 14})

plt.subplot(4, 1, 1)
plt.plot(t, sample, color='#1f77b4', linewidth=1.5)
plt.title("Sample", fontsize=16)
plt.grid(True, alpha=0.3)
plt.ylim(-0.5, 0.5)

plt.subplot(4, 1, 2)
plt.title("(1) Windowing", fontsize=16)
plt.plot(t, sample, color='#1f77b4', alpha=0.15, linewidth=1)

for i, (window, pos) in enumerate(zip(normalized_windows, window_positions)):
    plt.plot(t[pos:pos+window_size], window, color='#1f77b4', linewidth=1.5)

plt.grid(True, alpha=0.3)
plt.xlim(0, 1000)
plt.ylim(-2.5, 2.5)

plt.subplot(4, 1, 3)
plt.title("(2) Discretisation", fontsize=16)
plt.xlim(0, 1000)
plt.ylim(0, 1)
plt.axis('off')

plt.text(100, 0.5, "bca", fontsize=14, color='#1f77b4')
plt.text(250, 0.5, "bbb", fontsize=14, color='#1f77b4')
plt.text(450, 0.5, "bbb", fontsize=14, color='#1f77b4')
plt.text(650, 0.5, "bbb", fontsize=14, color='#1f77b4')
plt.text(850, 0.5, "bbb", fontsize=14, color='#1f77b4')

plt.subplot(4, 1, 4)
plt.title("(3) Bag-of-Patterns model", fontsize=16)
plt.bar(range(len(patterns)), pattern_counts, color='#1f77b4')
plt.xticks(range(len(patterns)), patterns)
plt.ylabel("Counts")
plt.grid(True, alpha=0.3)

plt.tight_layout(pad=1.5)
plt.savefig('images/chap14_images/fig_dictionary_approach.png', dpi=150, bbox_inches='tight')
plt.show()


## Dictionary Classification
*Block: dictionary-classification — run for real (was eval: false)*

In [ ]:
%%time
from aeon.classification.dictionary_based import BOSSEnsemble, WEASEL_V2, TemporalDictionaryEnsemble

boss = BOSSEnsemble(
    max_ensemble_size=5,
    min_window=60,
    random_state=3542,
    n_jobs=10
)

weasel = WEASEL_V2(
    random_state=3542,
    n_jobs=10
)

tde = TemporalDictionaryEnsemble(
    n_parameter_samples=250,
    max_ensemble_size=50,
    randomly_selected_params=True,
    random_state=3542,
    n_jobs=10
)

_start = time.time()
boss.fit(X_train, y_train)
timings["BOSS"] = time.time() - _start
boss_preds = boss.predict(X_test)
results["BOSS"] = accuracy_score(y_test, boss_preds)
print(f"BOSS accuracy: {results['BOSS']:.4f}, time: {timings['BOSS']:.2f}s")

_start = time.time()
weasel.fit(X_train, y_train)
timings["WEASEL_V2"] = time.time() - _start
weasel_preds = weasel.predict(X_test)
results["WEASEL_V2"] = accuracy_score(y_test, weasel_preds)
print(f"WEASEL_V2 accuracy: {results['WEASEL_V2']:.4f}, time: {timings['WEASEL_V2']:.2f}s")

_start = time.time()
tde.fit(X_train, y_train)
timings["TDE"] = time.time() - _start
tde_preds = tde.predict(X_test)
results["TDE"] = accuracy_score(y_test, tde_preds)
print(f"TDE accuracy: {results['TDE']:.4f}, time: {timings['TDE']:.2f}s")


## Table 14.5 — Dictionary Classifier Comparison
*Block: tbl-dictionary-results*

In [ ]:
dict_preds = {
    "BOSS": boss_preds,
    "WEASEL V2": weasel_preds,
    "TDE": tde_preds,
}
tbl5 = build_summary_table(dict_preds, y_test, {
    "BOSS": timings.get("BOSS"), "WEASEL V2": timings.get("WEASEL_V2"), "TDE": timings.get("TDE")
})
print("\nTable 14.5: Dictionary classifier comparison")
print(tbl5.to_markdown(index=False))


## fig-dictionary-classifiers-comparison

In [ ]:
selected_keys = ['Euclidean', 'WDTW', 'ProximityForest (tuned)', 'catch22class',
                 'RDST', 'BOSS', 'WEASEL_V2', 'TDE']
results_filtered = {k: results[k] for k in selected_keys if k in results}

plt.figure(figsize=(10, 6))
plt.bar(results_filtered.keys(), results_filtered.values(), color=custom_palette[:len(results_filtered)])
plt.ylabel("Accuracy", fontsize=18)
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)

for i, (method, accuracy) in enumerate(results_filtered.items()):
    plt.text(i, accuracy + 0.02, f"{accuracy:.4f}", ha='center', fontsize=18)

plt.xticks(fontsize=18)
plt.xticks(rotation=45, ha='right', fontsize=16)
plt.yticks(fontsize=18)
plt.tight_layout()
plt.savefig('images/chap14_images/fig_dictionary_classifiers_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## Kernel Classification (ROCKET / MiniROCKET)
*Block: kernel-classification*

In [ ]:
%%time
from aeon.classification.convolution_based import RocketClassifier, MiniRocketClassifier
from sklearn.linear_model import RidgeClassifierCV

rocket = RocketClassifier(
    n_kernels=10000,
    estimator=RidgeClassifierCV(
        alphas=np.logspace(-3, 3, 11)
    ),
    class_weight=None,
    n_jobs=10,
    random_state=3542
)

mini_rocket = MiniRocketClassifier(
    n_kernels=10000,
    max_dilations_per_kernel=32,
    estimator=RidgeClassifierCV(
        alphas=np.logspace(-3, 3, 11)
    ),
    class_weight=None,
    random_state=3542
)

_start = time.time()
rocket.fit(X_train, y_train)
timings["ROCKET"] = time.time() - _start
rocket_preds = rocket.predict(X_test)
results["ROCKET"] = accuracy_score(y_test, rocket_preds)
print(f"ROCKET accuracy: {results['ROCKET']:.4f}, time: {timings['ROCKET']:.2f}s")

_start = time.time()
mini_rocket.fit(X_train, y_train)
timings["MiniROCKET"] = time.time() - _start
mini_rocket_preds = mini_rocket.predict(X_test)
results["MiniROCKET"] = accuracy_score(y_test, mini_rocket_preds)
print(f"MiniROCKET accuracy: {results['MiniROCKET']:.4f}, time: {timings['MiniROCKET']:.2f}s")


## Table 14.6 — Kernel Classifier Comparison
*Block: tbl-kernel-results*

In [ ]:
kernel_preds = {
    "ROCKET": rocket_preds,
    "MiniROCKET": mini_rocket_preds,
}
tbl6 = build_summary_table(kernel_preds, y_test, timings)
print("\nTable 14.6: Kernel classifier comparison")
print(tbl6.to_markdown(index=False))


## fig-kernel-classifiers-comparison

In [ ]:
selected_keys = ['Euclidean', 'WDTW', 'ProximityForest (tuned)', 'catch22class',
                 'RDST', 'WEASEL_V2', 'ROCKET', 'MiniROCKET']
results_filtered = {k: results[k] for k in selected_keys if k in results}

plt.figure(figsize=(10, 6))
plt.bar(results_filtered.keys(), results_filtered.values(), color=custom_palette[:len(results_filtered)])
plt.ylabel("Accuracy", fontsize=18)
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)

for i, (method, accuracy) in enumerate(results_filtered.items()):
    plt.text(i, accuracy + 0.02, f"{accuracy:.4f}", ha='center', fontsize=18)

plt.xticks(fontsize=18)
plt.xticks(rotation=45, ha='right', fontsize=16)
plt.yticks(fontsize=18)
plt.tight_layout()
plt.savefig('images/chap14_images/fig_kernel_classifiers_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## ResNet Classification
*Block: resnet-classification — run for real (was eval: false)*

In [ ]:
%%time
from aeon.classification.deep_learning import ResNetClassifier

n_epochs = 200
batch_size = 64
random_state = 3542
n_jobs = 10
verbose = False

resnet = ResNetClassifier(
    n_residual_blocks=3,
    n_conv_per_residual_block=3,
    n_filters=[128, 64, 64],
    kernel_size=[8, 5, 3],
    activation='relu',
    n_epochs=n_epochs,
    batch_size=batch_size,
    random_state=random_state,
    verbose=verbose
)

_start = time.time()
resnet.fit(X_train, y_train)
timings["ResNet"] = time.time() - _start
resnet_preds = resnet.predict(X_test)
results["ResNet"] = accuracy_score(y_test, resnet_preds)
print(f"ResNet accuracy: {results['ResNet']:.4f}, time: {timings['ResNet']:.2f}s")


## InceptionTime Classification
*Block: inceptiontime-classification — run for real (was eval: false)*

In [ ]:
%%time
from aeon.classification.deep_learning import InceptionTimeClassifier

inception_time = InceptionTimeClassifier(
    n_classifiers=3,
    n_epochs=n_epochs,
    batch_size=batch_size,
    random_state=random_state,
    verbose=verbose
)

_start = time.time()
inception_time.fit(X_train, y_train)
timings["InceptionTime"] = time.time() - _start
inception_time_preds = inception_time.predict(X_test)
results["InceptionTime"] = accuracy_score(y_test, inception_time_preds)
print(f"InceptionTime accuracy: {results['InceptionTime']:.4f}, time: {timings['InceptionTime']:.2f}s")


## H-InceptionTime Classification
*Block: h-inceptiontime-classification — run for real (was eval: false)*

**Note:** The QMD uses `IndividualInceptionClassifier` for H-InceptionTime. This is a single-network inception model; it is **not** the same as `HInceptionTimeClassifier` if one exists in your version of `aeon`. Check `aeon.classification.deep_learning` for `HInceptionTimeClassifier` and substitute if available for true H-InceptionTime behaviour.

In [ ]:
%%time
from aeon.classification.deep_learning import IndividualInceptionClassifier

h_inception = IndividualInceptionClassifier(
    n_epochs=n_epochs,
    batch_size=batch_size,
    random_state=random_state,
    verbose=verbose
)

_start = time.time()
h_inception.fit(X_train, y_train)
timings["H-InceptionTime"] = time.time() - _start
h_inception_preds = h_inception.predict(X_test)
results["H-Inception"] = accuracy_score(y_test, h_inception_preds)
print(f"H-InceptionTime accuracy: {results['H-Inception']:.4f}, time: {timings['H-InceptionTime']:.2f}s")


## LiteTime Classification
*Block: litetime-classification — run for real (was eval: false)*

In [ ]:
%%time
from aeon.classification.deep_learning import LITETimeClassifier

lite_time = LITETimeClassifier(
    n_classifiers=3,
    n_epochs=n_epochs,
    batch_size=batch_size,
    random_state=random_state,
    verbose=verbose
)

_start = time.time()
lite_time.fit(X_train, y_train)
timings["LiteTime"] = time.time() - _start
lite_time_preds = lite_time.predict(X_test)
results["LITETime"] = accuracy_score(y_test, lite_time_preds)
print(f"LiteTime accuracy: {results['LITETime']:.4f}, time: {timings['LiteTime']:.2f}s")


## Table 14.7 — Deep Learning Classifier Comparison

In [ ]:
dl_preds = {
    "ResNet": resnet_preds,
    "InceptionTime": inception_time_preds,
    "H-InceptionTime": h_inception_preds,
    "LiteTime": lite_time_preds,
}
tbl7 = build_summary_table(dl_preds, y_test, {
    "ResNet": timings.get("ResNet"),
    "InceptionTime": timings.get("InceptionTime"),
    "H-InceptionTime": timings.get("H-InceptionTime"),
    "LiteTime": timings.get("LiteTime"),
})
print("\nTable 14.7: Deep learning classifier comparison")
print(tbl7.to_markdown(index=False))


## Final Summary — All Tables + JSON Export

In [ ]:
os.makedirs('data/chapter14', exist_ok=True)

all_tables = {}
for name, tbl in [
    ("Table 14.1", tbl1),
    ("Table 14.2", tbl2),
    ("Table 14.3", tbl3),
    ("Table 14.4", tbl4),
    ("Table 14.5", tbl5),
    ("Table 14.6", tbl6),
    ("Table 14.7", tbl7),
]:
    all_tables[name] = tbl.to_dict('records')

with open('data/chapter14/classification_results.json', 'w') as f:
    json.dump(all_tables, f, indent=2)

print("All tables saved to data/chapter14/classification_results.json")

for name, records in all_tables.items():
    df = pd.DataFrame(records)
    print(f"\n{name}")
    print(df.to_markdown(index=False))
